In [ ]:
# training.ipynb
# Stage 2 of 2: the full sweep via the supervisor. Run AFTER
# prepare_training.ipynb (which builds the H5 AND smoke-validates the training
# pipeline), so this notebook is just the multi-hour run + resume.
REPO = "/workspace/stable-query-latent"
URL = "https://github.com/Nice9Tian/stable-query-latent.git"
LOG = "/workspace/stable_query_latent_logs/pipeline.log"
print('repo:', REPO)
print('log :', LOG)

In [ ]:
# Clone or pull before every run.
import os

if not os.path.isdir(os.path.join(REPO, ".git")):
    !git clone {URL} {REPO}

%cd {REPO}
!git remote set-url origin {URL}
!git remote -v
!git pull origin main
!git status --short
!git rev-parse HEAD


In [ ]:
# Start GPU + CPU + RAM + disk I/O monitor.
import subprocess, threading, time, psutil
from pathlib import Path

stop = False

def _read_int(path):
    try:
        text = Path(path).read_text().strip()
        if text == "max":
            return None
        return int(text)
    except Exception:
        return None

def get_memory_status():
    limit = _read_int("/sys/fs/cgroup/memory.max")
    used = _read_int("/sys/fs/cgroup/memory.current")
    if limit is None or used is None:
        limit = _read_int("/sys/fs/cgroup/memory/memory.limit_in_bytes")
        used = _read_int("/sys/fs/cgroup/memory/memory.usage_in_bytes")
    if limit and used and limit < 10**18:
        return used / limit * 100, used / 1024**3, limit / 1024**3, "cgroup"
    vm = psutil.virtual_memory()
    return vm.percent, vm.used / 1024**3, vm.total / 1024**3, "host"

def get_gpu_status():
    try:
        out = subprocess.run(
            ["nvidia-smi", "--query-gpu=utilization.gpu,memory.used,memory.total", "--format=csv,noheader,nounits"],
            capture_output=True,
            text=True,
            timeout=3,
        ).stdout.strip()
        parts = [p.strip() for p in out.splitlines()[0].split(",")]
        return f"{float(parts[0]):.0f}%, {float(parts[1])/1024:.1f}/{float(parts[2])/1024:.1f} GiB"
    except Exception as e:
        return f"n/a ({e})"

def monitor(interval=5):
    last_disk = psutil.disk_io_counters()
    last_t = time.time()
    psutil.cpu_percent(interval=None)
    while not stop:
        gpu = get_gpu_status()
        cpu = psutil.cpu_percent(interval=None)
        ram_pct, ram_used, ram_total, ram_source = get_memory_status()
        now_disk = psutil.disk_io_counters()
        now_t = time.time()
        dt = max(now_t - last_t, 1e-6)
        read_mb = (now_disk.read_bytes - last_disk.read_bytes) / 1e6 / dt
        write_mb = (now_disk.write_bytes - last_disk.write_bytes) / 1e6 / dt
        last_disk, last_t = now_disk, now_t
        print(
            f"[gpu] {gpu} | [cpu] {cpu:.0f}% | "
            f"[ram:{ram_source}] {ram_pct:.0f}% ({ram_used:.1f}/{ram_total:.1f} GiB) | "
            f"[disk] R {read_mb:.1f} MB/s W {write_mb:.1f} MB/s",
            flush=True,
        )
        time.sleep(interval)

threading.Thread(target=monitor, daemon=True).start()


In [ ]:
# (Optional) Inspect the memory plan before the full run. Calibrates (C,R) on
# pseudo batches, then prints per combo the chosen backward-mode + stem chunk
# size. Safe to skip -- the supervisor calibrates + plans automatically.
# (After this, you may set memory.calib: load in sweep.yaml to reuse calib.json.)
!python -u VICReg_review/oom_proxy.py \
  --h5 game_review_data/embedding_h5.h5 \
  --calib-out VICReg_review/heads/cloud_full_sweep_a100/calib.json --measure
!python -u -m VICReg_review.sweep.planner \
  --config VICReg_review/sweep/sweep.yaml \
  --calib-json VICReg_review/heads/cloud_full_sweep_a100/calib.json


In [ ]:
# Train the full sweep via the NEW supervisor (clean redesign; replaces sweep_cloud).
# Reads VICReg_review/sweep/sweep.yaml. Spawns a persistent worker that
# calibrates once (pseudo-batch C,R), then trains every combo with the chunked
# stem -> FULL data, NO sentence cap. Crash-isolated: a worker OOM/crash makes
# the supervisor shrink the stem chunk, restart the worker, and resume from the
# ledger; one bad combo is marked 'failed' and never aborts the run.
# Resume: just re-run this cell -- combos verified done (vs checkpoint) are skipped.
# Probes are emitted to the queue and drained later in eval.ipynb.
# State: VICReg_review/heads/cloud_full_sweep_a100/{ledger.jsonl, calib.json, sweep_jobs/}
!python -u VICReg_review/sweep/supervisor.py \
  --config VICReg_review/sweep/sweep.yaml \
  --logout-address {LOG}


In [ ]:
# Training done. Stop the monitor and print the ledger summary.
# Probe draining + final eval + artifact collection live in eval.ipynb
# (run it next, on this same pod, after training finishes).
stop = True

import json
from collections import Counter
from pathlib import Path

out_dir = Path(REPO, 'VICReg_review/heads/cloud_full_sweep_a100')
ledger = out_dir / 'ledger.jsonl'
queue_dir = out_dir / 'probe_queue'

if ledger.exists():
    latest = {}
    for line in ledger.read_text(encoding='utf-8').splitlines():
        line = line.strip()
        if not line:
            continue
        try:
            rec = json.loads(line)
        except json.JSONDecodeError:
            continue
        if rec.get('combo_id'):
            latest[rec['combo_id']] = rec
    counts = Counter(r.get('status', '?') for r in latest.values())
    print(f'ledger: {dict(counts)} ({len(latest)} combos)')
    failed = [c for c, r in latest.items() if r.get('status') == 'failed']
    if failed:
        print(f'failed combos ({len(failed)}):', failed)
else:
    print('no ledger yet:', ledger)

pending = sorted(queue_dir.glob('*.json')) if queue_dir.exists() else []
print(f'probe jobs queued (drain these in eval.ipynb): {len(pending)}')
print('Next: run Pod/eval.ipynb on this pod to drain probes, run final eval, and archive.')
